In [1]:
import numpy as np
import scipy.stats as stats
from scipy.special import kolmogorov

n = 100
m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
k = len(m)
x = np.arange(0, k)


## H0: данные согласуются с законом равномерного распределения
## H1: ¬H0

### С помощью критерия χ²

In [2]:
P = 1 / k
Δ_est = np.sum((m - n * P) ** 2 / (n * P))

p_value = stats.chi2.sf(Δ_est, k - 1)
print(f"Δ_est = {Δ_est}")
print(f"p-value = {p_value:.7f}")


Δ_est = 16.4
p-value = 0.0589840


Поскольку p_value > α = 0.05, то нет веских оснований отвергнуть H0

### С помощью критерия Колмогорова

In [3]:
F_est = np.arange(0, k) / k


F_sample = np.concatenate([[0], np.cumsum(m)]) / n


Δ_est = np.sqrt(n) * max(
    [max(abs(F_sample[i] - F_est[i]), abs(F_sample[i + 1] - F_est[i])) for i in range(k)]
)

p_value = kolmogorov(Δ_est)
print(f"Δ_est = {Δ_est}")
print(f"p-value = {p_value}")


Δ_est = 1.4000000000000001
p-value = 0.039681879538114355


Поскольку p_value < α = 0.05, то отвергаем H0

### Сравнение
По критерию χ² не было веских оснований отвергнуть гипотезу H0, в то время как по критерию Колмогорова её отвергаеи

## H0: данные согласуются с законом нормального распределения
## H1: ¬H0

### С помощью критерия χ²

In [4]:
μ = np.sum(m * x) / n
σ = np.sqrt(np.sum((x - μ) ** 2 * m / n))
Φ = stats.norm.cdf(np.arange(1, 10), μ, σ)
P = np.concatenate((Φ, np.array([1]))) - np.concatenate((np.array([0]), Φ))

print(f"μ = {μ}, σ = {σ}")

print(f"{P = }")

Δ_est = np.sum((m - n * P) ** 2 / (n * P))
print(f"Δ_est = {Δ_est}")

p_value = stats.chi2.sf(Δ_est, 10 - 2 - 1)
print(f"p_value = {p_value}")


μ = 4.77, σ = 2.505414137423193
P = array([0.06619531, 0.06825332, 0.10549932, 0.13934648, 0.15727758,
       0.15169243, 0.12502207, 0.08805061, 0.05299025, 0.04567264])
Δ_est = 17.108920472354754
p_value = 0.016707182522531727


Поскольку p_value < α = 0.05, то отвергаем H0

### С помощью критерия Колмогорова

In [5]:
N = 50000


F_est = np.arange(0, k + 1) / k
F_sample = np.concatenate([[0], np.cumsum(m)]) / n


Δ_est = np.sqrt(n) * max(
    [max(abs(F_sample[i] - stats.norm.cdf(x[i], μ, σ)), abs(F_sample[i + 1] - stats.norm.cdf(x[i], μ, σ))) for i in range(k)]
)



delta_series = np.zeros(shape=N)
F_est_bootstrap = np.arange(0, n + 1) / n
for i in range(N):
    bootstrap_sample = stats.norm.rvs(μ, σ, n)
    sorted_sample = np.sort(bootstrap_sample)

    bootstrap_mu = np.mean(bootstrap_sample)
    bootstrap_sigma = np.std(bootstrap_sample) * n / (n - 1)


    cdf_values = stats.norm.cdf(sorted_sample, bootstrap_mu, bootstrap_sigma)
    diff1 = np.abs(cdf_values - F_est_bootstrap[:-1])
    diff2 = np.abs(cdf_values - F_est_bootstrap[1:])
    max_diff = np.max(np.maximum(diff1, diff2))
    delta_series[i] = np.sqrt(n) * max_diff

delta_series = np.sort(delta_series)


l = np.sum(delta_series >= Δ_est)
p_value = l / N

print(f"Δ_est = {Δ_est}")
print(f"p_value = {p_value}")


Δ_est = 1.013371112422481
p_value = 0.01226


Поскольку p_value < α = 0.05, то отвергаем H0

### Сравнение
Как по критерию χ², так и по критерию Колмогорова отвергаем H0

Поскольку p_value в случае Колмогорова меньше, то отвергаем уверенее